# 氏名漢字カナ突合モデル

漢字カナ突合モデルはJava17で記述されている。Mavenを用いて漢字カナ突合モデルをビルドする。
仕様等については[README](./README.md)を参照のこと。


## 必要なモジュール

- JDK 17
- Maven 3




## 事前準備
- バージョン
pom.xmlのバージョン番号は、../version.txtに記載

- ファイル
[dict](../dict)で作成した，辞書ファイル及び，[train](../train)で作成したAIモデルのファイルを，Javaのリソースとして使用するために，src/main/resources以下にファイルをコピーする。なお，事前に作成した辞書及び，事前学習済みのモデルを，[resources][https://kktg.digital.go.jp/support/resources/]で配布している。これらのファイルをあらかじめ，../dictおよび，../train/model, ../train/model_r以下にコピーしたのちに，下記スクリプトを実行してもよい。

In [1]:
import os
version="2.0o"

os.environ["VER"]=version

In [2]:
import glob
# latest date
for f in sorted(glob.glob(f"lib/kanjikana_core-{version}-*-jar-with-dependencies.jar")):
    jarfile = f

os.environ["JARFILE"]=jarfile

In [3]:
!echo $VER

2.0o


# 漢字とカナのマッチング

漢字姓名とカナ姓名が入力されたファイルをCSVもしくはTSV形式で用意する。漢字姓名とカナ姓名以外のフィールドが含まれていても良い。
なお，VMパラメタとして`-Xmx8196M`を指定し，8Gのメモリを確保すること。
### 実行クラス
jp.go.digital.kanjikana.core.executor.match.KanjiKanaMatchMain
### パラメタ
- infile
入力ファイル。行に漢字姓名とカナ姓名を含むファイルで，TSVもしくはCSV形式で作成しること。また，ヘッダがあっても良くヘッダがある場合には`has_header`パラメタを指定する。また，漢字姓名とカナ姓名の列を`kanji_idx`と`kana_idx`パラメタで指定する
- okfile
漢字姓名とカナ姓名が一致と判断されたレコードが保存されるファイル
- ngfile
漢字姓名とカナ姓名が不一致と判断されたレコードが保存されるファイル
- logfile
入力ファイルの各行を実行する際のログファイル
- kanji_idx
infileのファイルの漢字姓名の列番号，0から始まる
- kana_idx
infileのファイルのカナ姓名の列番号，0から始まる
- sep
infileの各列の区切り文字を指定する。csvもしくはtsvを指定する。デフォルトcsv
- thread_num
infileに大量の行が含まれる場合には，本プログラムをスレッドで多重化して計算することができる。`thread_num`に1を指定すると（デフォルト），シングルスレッドで実行され，それ以上を値を指定するとその数だけスレッドで計算される。スレッドでケインさんした場合には，`okfile`と`ngfile`，`logfile`の末尾にスレッド番号が付与されたファイルが作成される
- has_header
infileにヘッダ業がある場合には`true`を指定する。デフォルトは`false`であり，infileにヘッダ行は含まれない
- strategy
漢字姓名とカナ姓名の突合モデルをどれを使うかを指定する。BASIC, ONLY_AI, ONLY_DICT, AI, ENSEMBLEから選択し，セットする。    

  |モデル|内容|
  |-----|---|
  |ENSEMBLE||
  |BASIC|姓名辞書，異体字辞書との突合のみ|
  |ONLY_AI|AIモデルとの突合のみ|
  |ONLY_DICT|姓名辞書及び単漢字辞書，異体字辞書との突合のみ|
  |AI|BASIC + ONLY_AI|
  

In [4]:
# 出力結果がoutput.txtに出力される
!java -Xmx8196M -Dlog4j.configurationFile=log4j2.xml -classpath $JARFILE jp.go.digital.kanjikana.core.executor.match.KanjiKanaMatchMain --infile input.txt --outfile outfile.txt --kanji_idx 1 --kana_idx 2 --strategy ENSEMBLE --has_header true --thread_num 1 --sep csv


14:51:23,006 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Found logback-classic version ?
14:51:23,006 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Found logback-core version ?
14:51:23,007 |-WARN in ch.qos.logback.classic.LoggerContext[default] - Versions of logback-core and logback-classic are different!
14:51:23,007 |-INFO in ch.qos.logback.classic.util.ContextInitializer@351aa83f - No custom configurators were discovered as a service.
14:51:23,008 |-INFO in ch.qos.logback.classic.util.ContextInitializer@351aa83f - Trying to configure with ch.qos.logback.classic.util.DefaultJoranConfigurator
14:51:23,009 |-INFO in ch.qos.logback.classic.util.ContextInitializer@351aa83f - Constructed configurator of type class ch.qos.logback.classic.util.DefaultJoranConfigurator
14:51:23,011 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Could NOT find resource [logback-test.xml]
14:51:23,011 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Could NOT find

## 結果を表示する

In [5]:
!cat outfile.txt

no,kanji,kana,okng,start_date,end_date
1,山田　太郎,ヤマダ　ハナコ,ng,true,90,{true;山田;ヤマダ;WordEngine;DictReliableNormalized;1.0:/;}{true;太;ハナ;NandokuEngine;;1.0:/;}{true;郎;コ;NandokuEngine;;1.0:/;},2026-10-15 14:51:21.909 +0900,2026-10-15 14:51:21.957 +0900
2,山田　太郎,ヤマダタロウ,ok,true,90,{true;山;ヤマ;NandokuEngine;;1.0:/;}{true;太;ロウ;NandokuEngine;;1.0:/;},2026-10-15 14:51:21.962 +0900,2026-10-15 14:51:21.964 +0900
3,山田　太郎,サンダ　フトロウ,ok,true,90,{true;山;サン;NandokuEngine;;1.0:/;}{true;田;ダ;NandokuEngine;;1.0:/;}{true;太;フト;NandokuEngine;;1.0:/;}{true;郎;ロウ;NandokuEngine;;1.0:/;},2026-10-15 14:51:21.967 +0900,2026-10-15 14:51:21.970 +0900
4,BILLY JOEL,ビリー　ジョエル,ok,true,90,{true;ＢＩＬＬＹ;ビリー;FWordEngine;;1.0:/;0000000}{true;ＪＯＥＬ;ジョエル;FWordEngine;;1.0:/;0000000},2026-10-15 14:51:21.972 +0900,2026-10-15 14:51:22.049 +0900
5,BILLY JOEL,ビリー　ジュエル,ng,true,90,{true;ＢＩＬＬＹ;ビリー;FWordEngine;;1.0:/;0000000}{true;ＪＯＥＬ;ジュエル;FWordEngine;;1.0:/;0000000},2026-10-15 14:51:22.052 +0900,2026-10-15 14:51:22.108 +0900
6,BILLY JOEL,ビリージョエル,

# 漢字からカナを推測する
漢字姓名が入力されたファイルをCSVもしくはTSV形式で用意する。漢字姓名以外のフィールドが含まれていても良い。
なお，VMパラメタとして`-Xmx8196M`を指定し，8Gのメモリを確保すること。
### 実行クラス
jp.go.digital.kanjikana.core.executor.generate.Kanji2KanaMain
### パラメタ
- infile
入力ファイル。行に漢字姓名を含むファイルで，TSVもしくはCSV形式で作成しること。また，ヘッダがあっても良くヘッダがある場合には`has_header`パラメタを指定する。また，漢字姓名の列を`kanji_idx`パラメタで指定する
- outfile
漢字姓名から推測されたカナ姓名を出力したファイル
- n_best
カナ姓名の候補をいくつ出力するか、デフォルト5
- kanji_idx
infileのファイルの漢字姓名の列番号，0から始まる
- sep
infileの各列の区切り文字を指定する。csvもしくはtsvを指定する。デフォルトcsv
- has_header
infileにヘッダ業がある場合には`true`を指定する。デフォルトは`false`であり，infileにヘッダ行は含まれない


In [6]:
!java -Xmx8196M -Dlog4j.configurationFile=log4j2.xml -classpath $JARFILE jp.go.digital.kanjikana.core.executor.generate.Kanji2KanaMain --infile input.txt --outfile outfile_kana.txt --kanji_idx 1  --has_header true  --sep csv


14:51:35,382 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Found logback-classic version ?
14:51:35,382 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Found logback-core version ?
14:51:35,383 |-WARN in ch.qos.logback.classic.LoggerContext[default] - Versions of logback-core and logback-classic are different!
14:51:35,383 |-INFO in ch.qos.logback.classic.util.ContextInitializer@225a67e4 - No custom configurators were discovered as a service.
14:51:35,383 |-INFO in ch.qos.logback.classic.util.ContextInitializer@225a67e4 - Trying to configure with ch.qos.logback.classic.util.DefaultJoranConfigurator
14:51:35,384 |-INFO in ch.qos.logback.classic.util.ContextInitializer@225a67e4 - Constructed configurator of type class ch.qos.logback.classic.util.DefaultJoranConfigurator
14:51:35,386 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Could NOT find resource [logback-test.xml]
14:51:35,386 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Could NOT find

## 出力

In [7]:
!cat outfile_kana.txt

no,kanji,kana,okng,result,start_date,end_date
1,山田　太郎,ヤマダ　ハナコ,ng,kanji:山田　太郎;kanji1:山田　太郎;best:1;predict:ヤマダタロウ;probability:0.9994280608081346,2026-10-15 14:51:34.516 +0900,2026-10-15 14:51:38.275 +0900
1,山田　太郎,ヤマダ　ハナコ,ng,kanji:山田　太郎;kanji1:山田　太郎;best:2;predict:ヤマダヤスオ;probability:2.633688823272733E-4,2026-10-15 14:51:34.516 +0900,2026-10-15 14:51:38.275 +0900
1,山田　太郎,ヤマダ　ハナコ,ng,kanji:山田　太郎;kanji1:山田　太郎;best:3;predict:ヤマタタロウ;probability:1.244924491117243E-4,2026-10-15 14:51:34.516 +0900,2026-10-15 14:51:38.275 +0900
1,山田　太郎,ヤマダ　ハナコ,ng,kanji:山田　太郎;kanji1:山田　太郎;best:4;predict:ヤマダヤスロウ;probability:7.994594280297148E-5,2026-10-15 14:51:34.516 +0900,2026-10-15 14:51:38.275 +0900
1,山田　太郎,ヤマダ　ハナコ,ng,kanji:山田　太郎;kanji1:山田　太郎;best:5;predict:ヤマダダイロウ;probability:2.9957686466177286E-5,2026-10-15 14:51:34.516 +0900,2026-10-15 14:51:38.275 +0900
2,山田　太郎,ヤマダタロウ,ok,kanji:山田　太郎;kanji1:山田　太郎;best:1;predict:ヤマダタロウ;probability:0.9994280608081346,2026-10-15 14:51:38.288 +0900,2026-10-15 14:51:38.446 +0900
2,

## 出力の見方
kanji:山田　太郎;kanji1:山田　太郎;best:1;predict:ヤマダタロウ;probability:-2.270516720071963E-4,2025-10-06 15:59:29.910 +0900,2025-10-06 15:59:34.276 +0900

- kanji: input.txtの漢字姓名
- kanji1: 旧姓など複数の名前に分けられる場合の通し番号、 山田［田中】 花子 の場合には、kanji1:山田 花子、kanji2:田中 花子になる
- best:1  nbestで指定した数だけ出力。数字が小さいほど確率が高い
- predict: 予測した漢字の読みカナ
- probability: 確率、log(probability) で確率になる。 


# カナから漢字を推測する
カナ姓名が入力されたファイルをCSVもしくはTSV形式で用意する。カナ姓名以外のフィールドが含まれていても良い。
なお，VMパラメタとして`-Xmx8196M`を指定し，8Gのメモリを確保すること。
### 実行クラス
jp.go.digital.kanjikana.core.executor.generate.Kana2KanjiMain
### パラメタ
- infile
入力ファイル。行にカナ姓名を含むファイルで，TSVもしくはCSV形式で作成しること。また，ヘッダがあっても良くヘッダがある場合には`has_header`パラメタを指定する。また，カナ姓名の列を`kana_idx`パラメタで指定する
- outfile
カナ姓名から推測された漢字姓名を出力したファイル
- n_best
漢字姓名の候補をいくつ出力するか、デフォルト5
- kana_idx
infileのファイルのカナ姓名の列番号，0から始まる
- sep
infileの各列の区切り文字を指定する。csvもしくはtsvを指定する。デフォルトcsv
- has_header
infileにヘッダ業がある場合には`true`を指定する。デフォルトは`false`であり，infileにヘッダ行は含まれない


In [8]:
!java -Xmx8196M -Dlog4j.configurationFile=log4j2.xml -classpath $JARFILE jp.go.digital.kanjikana.core.executor.generate.Kana2KanjiMain --infile input.txt --outfile outfile_kanji.txt --kana_idx 2  --has_header true  --sep csv


14:51:45,333 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Found logback-classic version ?
14:51:45,333 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Found logback-core version ?
14:51:45,333 |-WARN in ch.qos.logback.classic.LoggerContext[default] - Versions of logback-core and logback-classic are different!
14:51:45,333 |-INFO in ch.qos.logback.classic.util.ContextInitializer@65210910 - No custom configurators were discovered as a service.
14:51:45,334 |-INFO in ch.qos.logback.classic.util.ContextInitializer@65210910 - Trying to configure with ch.qos.logback.classic.util.DefaultJoranConfigurator
14:51:45,334 |-INFO in ch.qos.logback.classic.util.ContextInitializer@65210910 - Constructed configurator of type class ch.qos.logback.classic.util.DefaultJoranConfigurator
14:51:45,336 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Could NOT find resource [logback-test.xml]
14:51:45,336 |-INFO in ch.qos.logback.classic.LoggerContext[default] - Could NOT find

In [9]:
!cat outfile_kanji.txt

no,kanji,kana,okng,result,start_date,end_date
1,山田　太郎,ヤマダ　ハナコ,ng,kana:ヤマダ　ハナコ;best:1;predict:山田華子;probability:0.6596129744795662,2026-10-15 14:51:44.439 +0900,2026-10-15 14:51:48.776 +0900
1,山田　太郎,ヤマダ　ハナコ,ng,kana:ヤマダ　ハナコ;best:2;predict:山田花子;probability:0.2311532436165189,2026-10-15 14:51:44.439 +0900,2026-10-15 14:51:48.776 +0900
1,山田　太郎,ヤマダ　ハナコ,ng,kana:ヤマダ　ハナコ;best:3;predict:やまだはなこ;probability:0.016859089560439536,2026-10-15 14:51:44.439 +0900,2026-10-15 14:51:48.776 +0900
1,山田　太郎,ヤマダ　ハナコ,ng,kana:ヤマダ　ハナコ;best:4;predict:山田英子;probability:0.01630778791909111,2026-10-15 14:51:44.439 +0900,2026-10-15 14:51:48.776 +0900
1,山田　太郎,ヤマダ　ハナコ,ng,kana:ヤマダ　ハナコ;best:5;predict:山田はな子;probability:0.012284870466467887,2026-10-15 14:51:44.439 +0900,2026-10-15 14:51:48.776 +0900
2,山田　太郎,ヤマダタロウ,ok,kana:ヤマダタロウ;best:1;predict:山田太郎;probability:0.9310088527213846,2026-10-15 14:51:48.786 +0900,2026-10-15 14:51:48.936 +0900
2,山田　太郎,ヤマダタロウ,ok,kana:ヤマダタロウ;best:2;predict:山田太朗;probability:0.047754688896577614,2026-10

## javadocを作成する
Javadocを作成し，/docs以下へ移動する。GithubPagesで見えるようにする。 

URLは [https://studious-chainsaw-5k2p5q6.pages.github.io/](https://studious-chainsaw-5k2p5q6.pages.github.io/)


In [10]:
!mvn javadoc:javadoc

[INFO] Scanning for projects...
[WARNING] 
[WARNING] Some problems were encountered while building the effective model for jp.go.digital.kanjikana:kanjikana_core:jar:2.0o
[WARNING] 'dependencies.dependency.(groupId:artifactId:type:classifier)' must be unique: net.sourceforge.argparse4j:argparse4j:jar -> duplicate declaration of version 0.9.0 @ line 58, column 21
[WARNING] 
[WARNING] It is highly recommended to fix these problems because they threaten the stability of your build.
[WARNING] 
[WARNING] For this reason, future Maven versions might no longer support building such malformed projects.
[WARNING] 
[INFO] 
[INFO] ---------------< jp.go.digital.kanjikana:kanjikana_core >---------------
[INFO] Building kanjikana_core 2.0o
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] >>> maven-javadoc-plugin:3.12.0:javadoc (default-cli) > generate-sources @ kanjikana_core >>>
[INFO] 
[INFO] <<< maven-javadoc-plugin:3.12.0:javadoc (default-cli) < gen

In [11]:
!date

2026年 10月 15日 木曜日 14:52:00 JST
